# Gene Regulatory Network Inference in High-Grade Serous Ovarian Cancer
## PANDA and LIONESS Analysis of TCGA-OV Data

**Author:** Aritra Mukherjee  
**Date:** May 2026  
**GitHub:** https://github.com/Aridoge13

---

### Biological Motivation

High-grade serous ovarian cancer (HGSOC) is characterised by extensive patient-to-patient heterogeneity in therapeutic response and metastatic behaviour. Critically, this heterogeneity is not fully explained by somatic mutation profiles alone — it is substantially driven by differences in gene regulatory architecture between patients. Standard differential expression analyses collapse this patient-level variation into population averages, masking the regulatory rewiring that underlies individual clinical trajectories.

This notebook applies two complementary network inference tools developed by the Kuijjer laboratory:

- **PANDA** (Passing Attributes between Networks for Data Assimilation): Reconstructs a consensus gene regulatory network by integrating three data sources — transcription factor (TF) binding motif priors, protein–protein interaction (PPI) networks, and gene co-expression patterns from bulk RNA-seq. PANDA uses a message-passing algorithm to iteratively reconcile these three sources into a single weighted bipartite network of TF→gene regulatory edges.

- **LIONESS** (Linear Interpolation to Obtain Network Estimates for Single Samples): Extends PANDA to the single-sample level. By applying a leave-one-out principle to the PANDA message-passing framework, LIONESS infers a separate regulatory network for each individual patient. This enables direct comparison of regulatory architectures across patients — a prerequisite for understanding how network rewiring correlates with clinical phenotypes such as platinum resistance or metastatic progression.

**Key question this notebook addresses:** Do HGSOC patients with different clinical outcomes (platinum-sensitive vs. platinum-resistant) differ in their patient-specific regulatory network architectures, and if so, which TF–gene regulatory edges are most consistently rewired?

---

## 0. Environment Setup

Install all required dependencies. NetZooPy is the Python implementation of the NetZoo suite of network biology tools, which includes PyPanda and LIONESS.

In [ ]:
# Install NetZooPy and dependencies
# Run this cell once, then restart the kernel
%pip install netzoopy pandas numpy matplotlib seaborn scipy scikit-learn pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# NetZooPy imports
from netZooPy.panda import Panda
from netZooPy.lioness import Lioness

print("All imports successful.")

---
## 1. Data Acquisition

### 1.1 Gene Expression Data (TCGA-OV)

Download from UCSC Xena: https://xenabrowser.net/datapages/

Navigate to: **TCGA Ovarian Cancer (OV)** → **Gene Expression RNAseq** → **IlluminaHiSeq pancan normalized**

File to download: `TCGA-OV.htseq_fpkm.tsv.gz`

Also download the **phenotype/clinical data**: `TCGA-OV.GDC_phenotype.tsv.gz`

Place both files in a `data/` subdirectory.

### 1.2 Prior Networks

PANDA requires two prior networks:
- **Motif prior**: TF binding site matrix (TFs × genes), derived from JASPAR motif scanning
- **PPI prior**: Protein–protein interaction network (TFs × TFs)

Pre-built human priors are available from the Kuijjer lab's SPONGE pipeline and from the NetZoo data repository. Download from:
https://netzoo.s3.us-east-2.amazonaws.com/netZooPy/tutorial_datasets/

Files needed:
- `motif_prior.txt` — TF–gene motif matrix
- `ppi_prior.txt` — TF–TF PPI network

**Note on priors:** These priors were generated using JASPAR 2022 motifs scanned against human promoter regions (−750 to +250 bp around TSS), filtered at a threshold of p < 10⁻⁴. The PPI network is derived from StringDB with a confidence score threshold of 0.9. Understanding how these priors are constructed is important: the motif prior encodes *potential* regulatory interactions based on TF binding affinity, while PANDA uses the expression data to determine which of these potential interactions are *active* in the biological context of interest.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

# Download pre-built human prior networks from NetZoo
!wget -P data/ https://netzoo.s3.us-east-2.amazonaws.com/netZooPy/tutorial_datasets/motif_prior.txt
!wget -P data/ https://netzoo.s3.us-east-2.amazonaws.com/netZooPy/tutorial_datasets/ppi_prior.txt

print("Prior networks downloaded.")

---
## 2. Data Preprocessing

### 2.1 Load and Inspect Gene Expression Data

In [ ]:
# Load TCGA-OV expression data
# UCSC Xena format: genes as rows, samples as columns
expr = pd.read_csv('data/TCGA-OV.htseq_fpkm.tsv.gz', sep='\t', index_col=0)

print(f"Expression matrix shape: {expr.shape}")
print(f"Number of genes: {expr.shape[0]}")
print(f"Number of samples: {expr.shape[1]}")
expr.head()

In [ ]:
# Load clinical/phenotype data
clinical = pd.read_csv('data/TCGA-OV.GDC_phenotype.tsv.gz', sep='\t', index_col=0)

print(f"Clinical data shape: {clinical.shape}")
print("\nAvailable clinical columns:")
print(clinical.columns.tolist())

### 2.2 Identify Platinum Response Groups

Platinum resistance is a central clinical challenge in HGSOC. We will stratify patients by their response to platinum-based chemotherapy to ask whether regulatory network architecture differs between sensitive and resistant patients.

**[YOUR INTERPRETATION — fill after running]:** After inspecting the clinical columns above, note which column encodes platinum/drug response and what the response categories are. Comment here on what proportion of patients are platinum-sensitive vs. resistant in this cohort.

In [ ]:
# Inspect columns related to drug response
response_cols = [col for col in clinical.columns if any(
    term in col.lower() for term in ['drug', 'response', 'platinum', 'treatment', 'therapy', 'measure']
)]
print("Potentially relevant columns:")
for col in response_cols:
    print(f"  {col}: {clinical[col].value_counts().to_dict()}")

In [ ]:
# Define platinum response groups
# Common TCGA encoding: 'Complete Remission/Response' = sensitive
#                        'Progressive Disease' or 'Stable Disease' = resistant
# Adjust the column name below based on what you found above

RESPONSE_COL = 'primary_diagnosis.treatment_or_therapy'  # UPDATE THIS based on your inspection

sensitive_samples = clinical[clinical[RESPONSE_COL].str.contains(
    'Complete|Partial|yes', case=False, na=False
)].index.tolist()

resistant_samples = clinical[clinical[RESPONSE_COL].str.contains(
    'Progressive|Stable|no', case=False, na=False
)].index.tolist()

print(f"Platinum-sensitive samples: {len(sensitive_samples)}")
print(f"Platinum-resistant samples: {len(resistant_samples)}")

### 2.3 Preprocess Expression Matrix for PANDA

PANDA expects:
- Genes as rows, samples as columns
- Gene identifiers matching those in the motif prior
- Log-transformed, normalised expression values

The UCSC Xena data is provided as log2(FPKM+1). We need to ensure gene IDs are in Hugo symbol format to match the motif prior.

In [ ]:
# Load prior networks and inspect gene/TF identifiers
motif = pd.read_csv('data/motif_prior.txt', sep='\t', header=None,
                    names=['TF', 'gene', 'score'])
ppi = pd.read_csv('data/ppi_prior.txt', sep='\t', header=None,
                  names=['TF1', 'TF2', 'score'])

print(f"Motif prior: {motif.shape[0]} TF-gene edges")
print(f"Unique TFs in motif prior: {motif['TF'].nunique()}")
print(f"Unique genes in motif prior: {motif['gene'].nunique()}")
print(f"\nPPI prior: {ppi.shape[0]} TF-TF edges")
print(f"\nExample motif prior entries:")
print(motif.head())

In [ ]:
# Filter expression matrix to genes present in motif prior
prior_genes = motif['gene'].unique()
expr_filtered = expr[expr.index.isin(prior_genes)]

print(f"Genes in motif prior: {len(prior_genes)}")
print(f"Genes in expression data: {expr.shape[0]}")
print(f"Overlapping genes (used for PANDA): {expr_filtered.shape[0]}")

# Remove low-variance genes (bottom 20%) - reduces noise
gene_variance = expr_filtered.var(axis=1)
variance_threshold = gene_variance.quantile(0.20)
expr_filtered = expr_filtered[gene_variance >= variance_threshold]

print(f"Genes after variance filtering: {expr_filtered.shape[0]}")

In [ ]:
# Save processed expression matrix for PANDA input
# PANDA expects: genes as rows, samples as columns, no header index issues
expr_filtered.to_csv('data/expr_panda_input.txt', sep='\t')

print(f"Final expression matrix saved: {expr_filtered.shape[0]} genes x {expr_filtered.shape[1]} samples")

---
## 3. PANDA: Consensus Gene Regulatory Network

PANDA reconstructs a **population-level consensus GRN** by reconciling three evidence sources through iterative message-passing:

1. **Motif prior** (W): Which TFs *can* bind which gene promoters (based on sequence)
2. **PPI network** (P): Which TFs interact at the protein level (co-regulatory potential)
3. **Co-expression** (C): Which genes are co-expressed (likely co-regulated)

The algorithm updates TF–gene edge weights iteratively until convergence, producing a weighted bipartite network where each edge weight reflects the *regulatory strength* between a TF and a target gene in the context of this expression dataset.

**Why this matters for HGSOC:** The consensus network tells us the aggregate regulatory landscape of ovarian cancer — which TFs are most broadly regulatory and which genes are under the tightest regulatory control. This provides the population-level baseline against which patient-specific deviations (LIONESS) are measured.

In [ ]:
# Run PANDA
# Parameters:
#   expression_file: genes x samples expression matrix
#   motif_file: TF-gene prior (3-column: TF, gene, score)
#   ppi_file: TF-TF PPI prior (3-column: TF1, TF2, score)
#   save_memory: True reduces RAM usage for large datasets
#   keep_expression_matrix: needed for LIONESS downstream

print("Running PANDA... (this may take 10-30 minutes depending on dataset size)")

panda_obj = Panda(
    expression_file='data/expr_panda_input.txt',
    motif_file='data/motif_prior.txt',
    ppi_file='data/ppi_prior.txt',
    save_memory=False,
    keep_expression_matrix=True,  # Required for LIONESS
    modeProcess='intersection'    # Use genes/TFs present in all three inputs
)

print("PANDA complete.")

In [ ]:
# Extract and save the PANDA network
# panda_network is a DataFrame: rows = TF-gene pairs, columns = [TF, gene, edge_weight]
panda_network = panda_obj.panda_network
panda_network.to_csv('results/panda_network.txt', sep='\t', index=False)

print(f"PANDA network shape: {panda_network.shape}")
print(f"Number of TF-gene edges: {len(panda_network)}")
print(f"\nTop 10 edges by regulatory weight:")
print(panda_network.nlargest(10, panda_network.columns[-1]))

### 3.1 Visualise the Consensus Network — Top Regulatory TFs

We summarise each TF's regulatory influence by summing its edge weights across all target genes. TFs with the highest total edge weight are the most broadly regulatory in the HGSOC context.

In [ ]:
# Summarise TF regulatory influence (sum of outgoing edge weights per TF)
tf_col = panda_network.columns[0]
gene_col = panda_network.columns[1]
weight_col = panda_network.columns[2]

tf_influence = panda_network.groupby(tf_col)[weight_col].sum().sort_values(ascending=False)

# Plot top 30 TFs
fig, ax = plt.subplots(figsize=(12, 6))
tf_influence.head(30).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 30 Transcription Factors by Regulatory Influence\n(PANDA consensus network, TCGA-OV)', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Transcription Factor', fontsize=11)
ax.set_ylabel('Summed Edge Weight', fontsize=11)
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig('figures/panda_top_tfs.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 10 regulatory TFs in HGSOC:")
print(tf_influence.head(10))

**[YOUR INTERPRETATION — fill after running]:**

Examine the top TFs identified by PANDA. Consider:
- Are any of these TFs known to be relevant to ovarian cancer biology? (e.g., TP53 is mutated in >96% of HGSOC)
- Are any associated with epithelial-to-mesenchymal transition (EMT), a key process in peritoneal metastasis?
- Cross-reference with known HGSOC transcription factor literature (e.g., FOXM1, E2F family, MYC are frequently implicated)

*Write your observations here after running the notebook.*

---
## 4. LIONESS: Patient-Specific Regulatory Networks

LIONESS extends PANDA to generate one network per patient. The mathematical intuition is elegant: for each sample *i*, LIONESS estimates what the PANDA network would look like if sample *i* were removed, then uses linear interpolation to infer sample *i*'s individual contribution to the aggregate network.

The result is a matrix of dimensions **(TF × gene edges) × patients**, where each column is a complete regulatory network for one patient. This is the key data structure that enables patient-stratified network analysis.

**Computational note:** LIONESS is memory-intensive. For the full TCGA-OV cohort (~300 samples), running LIONESS on a laptop may take several hours. Consider:
- Running on a subset (e.g., 50 sensitive + 50 resistant samples) for initial analysis
- Using GCP if available
- Running overnight

In [ ]:
# Optional: subset to platinum-sensitive and resistant samples only
# This reduces compute time and focuses the analysis on the biological question

# Find samples present in both expression data and clinical annotation
available_sensitive = [s for s in sensitive_samples if s in expr_filtered.columns]
available_resistant = [s for s in resistant_samples if s in expr_filtered.columns]

print(f"Available sensitive samples: {len(available_sensitive)}")
print(f"Available resistant samples: {len(available_resistant)}")

# Subset expression matrix
# Take up to 50 of each group to keep compute manageable
subset_samples = available_sensitive[:50] + available_resistant[:50]
expr_subset = expr_filtered[subset_samples]
expr_subset.to_csv('data/expr_subset_input.txt', sep='\t')

print(f"\nSubset expression matrix: {expr_subset.shape[0]} genes x {expr_subset.shape[1]} samples")

In [ ]:
# Run LIONESS
# LIONESS takes the PANDA object as input and iterates over samples
# Output: one network per sample, saved incrementally

print("Running LIONESS... (this will take significant time — consider running overnight)")

# Re-run PANDA on subset first
panda_subset = Panda(
    expression_file='data/expr_subset_input.txt',
    motif_file='data/motif_prior.txt',
    ppi_file='data/ppi_prior.txt',
    save_memory=False,
    keep_expression_matrix=True,
    modeProcess='intersection'
)

# Run LIONESS on the subset PANDA object
lioness_obj = Lioness(
    panda_subset,
    computing='cpu',       # Change to 'gpu' if CUDA available
    precision='single',    # Single precision reduces memory usage
    save_dir='results/',
    save_fmt='npy'         # Save as numpy binary for efficiency
)

print("LIONESS complete.")

In [ ]:
# Load LIONESS output
# LIONESS produces a matrix: rows = TF-gene edges, columns = samples
lioness_network = lioness_obj.export_lioness_table()

print(f"LIONESS network matrix shape: {lioness_network.shape}")
print(f"Rows = TF-gene edges: {lioness_network.shape[0]}")
print(f"Columns = patients: {lioness_network.shape[1] - 2}")  # subtract TF and gene columns
lioness_network.head()

---
## 5. Patient-Stratified Network Analysis

With patient-specific networks in hand, we can now ask: which TF–gene regulatory edges differ systematically between platinum-sensitive and platinum-resistant patients?

We use a two-sample t-test on each edge's weight distribution across the two patient groups. Edges with the largest and most statistically significant differences represent candidate regulatory mechanisms underlying platinum resistance.

In [ ]:
# Separate LIONESS networks by clinical group
# Identify sample columns in the LIONESS output
sample_cols = [col for col in lioness_network.columns if col not in ['tf', 'gene']]

sensitive_cols = [s for s in available_sensitive[:50] if s in sample_cols]
resistant_cols = [s for s in available_resistant[:50] if s in sample_cols]

print(f"Sensitive samples in LIONESS output: {len(sensitive_cols)}")
print(f"Resistant samples in LIONESS output: {len(resistant_cols)}")

# Extract edge weight matrices per group
sensitive_networks = lioness_network[sensitive_cols].values
resistant_networks = lioness_network[resistant_cols].values

In [ ]:
# Two-sample t-test for each TF-gene edge
# Tests whether edge weight differs significantly between groups
from scipy.stats import ttest_ind

t_stats = []
p_values = []
mean_diff = []

for i in range(sensitive_networks.shape[0]):
    t, p = ttest_ind(sensitive_networks[i], resistant_networks[i], equal_var=False)
    t_stats.append(t)
    p_values.append(p)
    mean_diff.append(sensitive_networks[i].mean() - resistant_networks[i].mean())

# Build results dataframe
results = lioness_network[['tf', 'gene']].copy()
results['t_stat'] = t_stats
results['p_value'] = p_values
results['mean_diff'] = mean_diff  # positive = higher in sensitive, negative = higher in resistant

# Multiple testing correction (Benjamini-Hochberg FDR)
from scipy.stats import false_discovery_control
results['fdr'] = false_discovery_control(results['p_value'].values)

# Sort by absolute mean difference
results['abs_mean_diff'] = results['mean_diff'].abs()
results_sorted = results.sort_values('abs_mean_diff', ascending=False)

# Save results
results_sorted.to_csv('results/differential_edges_sensitive_vs_resistant.txt', sep='\t', index=False)

print(f"Total edges tested: {len(results)}")
print(f"Significant edges (FDR < 0.05): {(results['fdr'] < 0.05).sum()}")
print(f"\nTop 20 differentially regulated TF-gene edges:")
print(results_sorted[['tf', 'gene', 'mean_diff', 'fdr']].head(20).to_string())

**[YOUR INTERPRETATION — fill after running]:**

Examine the top differentially regulated edges. Consider:
- Which TFs appear most frequently in the top edges? Are these known platinum-resistance regulators?
- Are edges higher in sensitive or resistant patients? What does a higher edge weight mean biologically (stronger TF–gene regulatory interaction)?
- Look up the top TFs in the literature: are FOXM1, E2F1, BRCA pathway regulators, or drug efflux-related TFs present?
- Note the FDR values — how many edges survive multiple testing correction?

*Write your observations here after running.*

### 5.1 Volcano Plot — Differential Edge Weights

In [ ]:
# Volcano plot: effect size (mean difference) vs. statistical significance
fig, ax = plt.subplots(figsize=(10, 7))

# Colour by significance
sig = results['fdr'] < 0.05
colors = np.where(sig, 
                  np.where(results['mean_diff'] > 0, '#d62728', '#1f77b4'),  # red=sensitive, blue=resistant
                  'lightgrey')

ax.scatter(results['mean_diff'], -np.log10(results['p_value']),
           c=colors, alpha=0.4, s=5, rasterized=True)

# Label top 10 edges
top10 = results_sorted.head(10)
for _, row in top10.iterrows():
    ax.annotate(f"{row['tf']}→{row['gene']}",
                (row['mean_diff'], -np.log10(row['p_value'])),
                fontsize=7, alpha=0.8)

ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
ax.set_xlabel('Mean Edge Weight Difference\n(Sensitive − Resistant)', fontsize=11)
ax.set_ylabel('-log₁₀(p-value)', fontsize=11)
ax.set_title('Differential TF–Gene Regulatory Edges\nPlatinum-Sensitive vs. Platinum-Resistant HGSOC (LIONESS)', 
             fontsize=12, fontweight='bold')

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', markersize=8, label='Higher in sensitive (FDR<0.05)'),
                   Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=8, label='Higher in resistant (FDR<0.05)'),
                   Line2D([0], [0], marker='o', color='w', markerfacecolor='lightgrey', markersize=8, label='Not significant')]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('figures/lioness_volcano.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 PCA of Patient-Specific Networks

Do platinum-sensitive and resistant patients occupy distinct regions of network space? PCA on the full LIONESS edge weight matrix reveals whether regulatory architecture alone can separate clinical groups.

In [ ]:
# PCA on patient-specific network edge weights
# Matrix: samples x edges
all_samples = sensitive_cols + resistant_cols
lioness_matrix = lioness_network[all_samples].T.values  # samples x edges

# Standardise
scaler = StandardScaler()
lioness_scaled = scaler.fit_transform(lioness_matrix)

# PCA
pca = PCA(n_components=10)
pca_coords = pca.fit_transform(lioness_scaled)

print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.3f}")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]:.3f}")
print(f"Variance explained by PC1+PC2: {pca.explained_variance_ratio_[:2].sum():.3f}")

In [ ]:
# Plot PCA
labels = ['Sensitive'] * len(sensitive_cols) + ['Resistant'] * len(resistant_cols)
colours = ['#d62728' if l == 'Sensitive' else '#1f77b4' for l in labels]

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(pca_coords[:, 0], pca_coords[:, 1],
                     c=colours, alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('PCA of Patient-Specific Regulatory Networks (LIONESS)\nTCGA-OV: Platinum-Sensitive vs. Resistant',
             fontsize=12, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#d62728', label='Platinum-sensitive'),
                   Patch(facecolor='#1f77b4', label='Platinum-resistant')]
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.savefig('figures/lioness_pca.png', dpi=150, bbox_inches='tight')
plt.show()

**[YOUR INTERPRETATION — fill after running]:**

Examine the PCA plot and consider:
- Do the two clinical groups cluster separately in network space, or do they overlap substantially?
- What does separation (or lack thereof) tell you about the relationship between regulatory architecture and platinum response in HGSOC?
- How much variance is captured by PC1 and PC2 combined? What does a low value suggest about the dimensionality of the regulatory heterogeneity?
- If groups don't separate cleanly, this is itself an interesting biological observation — what might cause within-group network heterogeneity?

*Write your observations here after running.*

### 5.3 Top Rewired TFs — Heatmap

In [ ]:
# Select top 20 most differentially regulated TFs (by mean absolute edge weight difference)
top_tfs = results_sorted.groupby('tf')['abs_mean_diff'].mean().nlargest(20).index.tolist()

# Filter LIONESS matrix to edges involving these TFs
top_edges_mask = lioness_network['tf'].isin(top_tfs)
top_lioness = lioness_network[top_edges_mask]

# Summarise: mean edge weight per TF per sample
tf_sample_matrix = top_lioness.groupby('tf')[all_samples].mean()

# Add group labels for column colors
col_colors = pd.Series(
    ['#d62728'] * len(sensitive_cols) + ['#1f77b4'] * len(resistant_cols),
    index=all_samples
)

# Heatmap
fig = plt.figure(figsize=(16, 8))
g = sns.clustermap(
    tf_sample_matrix,
    col_colors=col_colors,
    cmap='RdBu_r',
    center=0,
    xticklabels=False,
    yticklabels=True,
    figsize=(16, 8),
    dendrogram_ratio=0.1,
    cbar_pos=(0.02, 0.8, 0.03, 0.15)
)
g.fig.suptitle('Top 20 Rewired TFs: Patient-Specific Network Edge Weights\n(Red = Sensitive, Blue = Resistant)',
               y=1.02, fontsize=12, fontweight='bold')
plt.savefig('figures/lioness_heatmap_top_tfs.png', dpi=150, bbox_inches='tight')
plt.show()

**[YOUR INTERPRETATION — fill after running]:**

Examine the heatmap and consider:
- Do samples cluster by clinical group, or is there substantial mixing? What does this tell you about regulatory heterogeneity?
- Which TFs show the most consistent differential regulation across the group? These are the strongest candidates for functional follow-up.
- Are there subgroups *within* the sensitive or resistant patients that show distinct regulatory patterns? This kind of within-group heterogeneity is biologically important in HGSOC.
- Look up 2-3 of the top TFs in GeneCards or PubMed. Are they known in ovarian cancer? In drug resistance?

*Write your observations here after running.*

---
## 6. Summary and Next Steps

**[YOUR INTERPRETATION — fill after running the full notebook]:**

Summarise what you found:
1. What does the PANDA consensus network tell you about the aggregate regulatory landscape of HGSOC?
2. Do LIONESS patient-specific networks separate platinum-sensitive from resistant patients?
3. Which TF–gene edges are most consistently rewired between the two groups?
4. What are the biological implications of your top findings?

**Natural extensions of this analysis:**
- Apply PORCUPINE to quantify pathway-level regulatory heterogeneity across patients
- Apply SCORPION to the scRNA-seq ovarian cancer data to resolve regulatory heterogeneity at single-cell resolution
- Integrate MARMOT for multi-omics-aware single-sample network inference
- Validate top TF candidates against the Oncosys-OVA longitudinal dataset (primary vs. relapsed tumour samples)

---

**Technical environment:**

In [ ]:
import sys, netZooPy
print(f"Python: {sys.version}")
print(f"NetZooPy: {netZooPy.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")